# 1. The Simple Chain (prompt | model | parser)

The most fundamental chain — and the one you'll write most often. Three steps: build a prompt, call
the model, parse the output. Master this and every bigger chain is just more of the same.

---

## 1. Simple Definition

> **Kid version:** It's a three-step sandwich machine. You put in a filling (your topic), the machine
> **writes the order** (prompt), **makes the sandwich** (model), and **wraps it neatly** (parser).
> Out comes a ready sandwich (clean text).

**Professional definition:** A simple chain is a `RunnableSequence` of three stages — a **prompt
template** (formats input into a prompt), a **chat model** (generates a response), and an **output
parser** (converts the model's message into a usable Python value) — connected with `|`.

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Write a one-line tip about {topic}.")
chain = prompt | model | StrOutputParser()

chain.invoke({"topic": "sleep"})     # "Keep a consistent bedtime to improve sleep quality."
```

---

## 2. Why Does It Exist?

**The problem:** Even the simplest LLM call has three chores: turn your variables into a prompt, call
the model, and pull clean data out of the model's message object. Doing them separately is verbose and
repetitive.

### Before (manual, three separate calls)

```python
messages = prompt.format_messages(topic="sleep")   # 1) format
ai_msg = model.invoke(messages)                     # 2) call model
text = ai_msg.content                               # 3) unwrap .content
```

### After (one chain)

```python
chain = prompt | model | StrOutputParser()
text = chain.invoke({"topic": "sleep"})             # all three, one call
```

The chain packages the three chores into one reusable object that also supports **streaming,
batching, and async** automatically.

---

## 3. Real-Life Analogy

**A coffee shop order** ☕. You say a flavor (input). The barista **writes the order on a cup**
(prompt), **makes the drink** (model), and **puts a lid on it** (parser) so you can actually carry it.
Each role is distinct, and together they turn "vanilla" into a drinkable coffee.

---

## 4. Where It Fits in LangChain Architecture

```
   {"topic": "sleep"}            ← dict input
        │
        ▼
   PromptTemplate   ──► PromptValue / messages
        │
        ▼
   ChatModel        ──► AIMessage (has .content, maybe .tool_calls)
        │
        ▼
   OutputParser     ──► clean Python value (str, list, dict, object)
        │
        ▼
   final output
```

This prompt → model → parser trio is the atom of LangChain. Sequential, parallel, and conditional
chains are built by combining these atoms.

---

## 5. Internal Working — data flowing through

```
  chain = prompt | model | parser
  chain.invoke({"topic": "sleep"})

  ① {"topic": "sleep"}
        │  prompt fills the template
        ▼
  ② ChatPromptValue → [HumanMessage("Write a one-line tip about sleep.")]
        │  model generates
        ▼
  ③ AIMessage(content="Keep a consistent bedtime...")
        │  parser extracts
        ▼
  ④ "Keep a consistent bedtime..."   ← a plain str, ready to use
```

Notice how the **type changes** at each step: dict → messages → AIMessage → str. Each stage's output
type must fit the next stage's expected input — the trio above is the canonical, always-valid flow.

---

## 6. The three parts

### `PromptTemplate` (step 1)

**Definition:** Turns input variables into a formatted prompt/messages.

**Why it exists:** Separates the reusable prompt *shape* from the *values* you fill in.

**When developers use it:** The entrance of almost every chain.

**Real-life use case:** Writing the order on the cup.

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    ("human", "Write a one-line tip about {topic}."),
])
```

---

### `ChatModel` (step 2)

**Definition:** The LLM that generates a response message.

**Why it exists:** It's the actual intelligence producing the answer.

**When developers use it:** The middle of every chain.

**Real-life use case:** The barista making the drink.

```python
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
```

---

### `OutputParser` (step 3)

**Definition:** Converts the model's `AIMessage` into a usable value (string, list, dict, Pydantic
object).

**Why it exists:** Your program wants clean data, not a message object. (See your Output Parser notes.)

**When developers use it:** Whenever you need the result in a specific shape; `StrOutputParser` is the
most common (just the text).

**Real-life use case:** Putting a lid on so you can carry the coffee.

```python
from langchain_core.output_parsers import StrOutputParser
StrOutputParser()          # AIMessage → its .content string
```

---

## 7. It's a Runnable — so you get everything for free

Because the chain is itself a Runnable, all these work out of the box:

```python
chain.invoke({"topic": "sleep"})                 # one input

chain.batch([{"topic": "sleep"}, {"topic": "water"}])   # many inputs, parallelized

for chunk in chain.stream({"topic": "sleep"}):   # stream tokens as they arrive
    print(chunk, end="", flush=True)

await chain.ainvoke({"topic": "sleep"})          # async
```

You wrote three components with `|`, and got batching, streaming, and async on the whole pipeline.

---

## 8. Can a chain be even simpler?

Yes — a chain doesn't *require* all three parts. Two-step chains are common:

```python
# prompt | model  → returns the raw AIMessage (no parser)
(prompt | model).invoke({"topic": "sleep"}).content

# model | parser  → skip templating, feed messages directly
```

But `prompt | model | parser` is the standard, complete simple chain — start here.

---

## A quick note on the legacy `LLMChain`

Old tutorials use `LLMChain(llm=model, prompt=prompt)`. It did roughly what `prompt | model` does. It
**still works but is deprecated** — prefer LCEL (`|`) for new code. Recognize it in old code; don't
write it.

```python
# Legacy (deprecated) — shown only so you recognize it:
# from langchain.chains import LLMChain
# chain = LLMChain(llm=model, prompt=prompt)

# Modern equivalent:
chain = prompt | model | StrOutputParser()
```

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
# step1: create the prompt template
prompt = PromptTemplate(template='Generate 5 interesting facts about {topic}',
                        input_variables=['topic'])

In [4]:
# step2: Initialize the language model
llm = ChatOllama(model="qwen3:8b")

In [5]:
# step3: Initialize output parser
output_parser = StrOutputParser()

In [6]:
# step4: Create the chain
chain = prompt | llm| output_parser

In [7]:
# invoke the chain with a topic
result = chain.invoke({'topic':'cricket'})

In [8]:
print(result)

1. **The First Cricket Club**: The Hambledon Cricket Club, founded in 1760 in England, is widely regarded as the first organized cricket club. It played a pivotal role in formalizing rules and promoting the sport, laying the groundwork for modern cricket.

2. **Test Match Origins**: The term "Test Match" comes from the 1877 Ashes series between England and Australia. It was introduced to describe matches that tested players' skills over long periods, eventually becoming the longest format of the game (five days).

3. **The Cricket Ball's Unique Design**: A standard cricket ball is made of leather with a cork core and is stitched with 86 pieces of leather. The seams are critical for seam bowlers, who exploit them to create movement and variation in the ball's flight.

4. **The Duckworth-Lewis Method**: This mathematical formula, developed in the 1990s, is used to calculate fair targets in rain-affected limited-overs matches. It ensures matches remain competitive even when interrupted, b

In [9]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
